# pygbif Example: GBIF Occurrence Data

GBIF provides the [pygbif](https://pygbif.readthedocs.io/en/latest/index.html) Python package for querying occurrence data. This example uses the package to perform a simple query.

1. Load dependencies and turn on caching

In [60]:
import pygbif
from pygbif import occurrences
from pygbif import species
from pygbif import maps


pygbif.caching(True, expire_after=3600)

{'cache': True,
 'name': '/var/folders/h4/kmjvt49974l5xmpddksqtvgm0000gn/T/pygbif_requests_cache',
 'backend': 'sqlite',
 'expire_after': 3600,
 'allowable_codes': (200,),
 'allowable_methods': ('GET',)}

2. Lookup the taxonomy Id for Pumas

In [ ]:
# search for scienctific names like "Puma"
puma_types = species.name_lookup(q="Puma", rank="species")
print(f"Found {puma_types['count']} Puma-like names")

3. Count the occurrence data for each Puma-like species in 2015

In [ ]:
puma_occurrences = {}
for puma in puma_types['results']:
    sci_name = puma['scientificName']
    taxon_key = puma['key']
    num_occurrences = occurrences.count(taxonKey=taxon_key, year=2015)
    if num_occurrences > 0:
        puma_occurrences[taxon_key] = {
            'scientificName': sci_name,
            'numOccurrences': num_occurrences
        }
print(f"Found {len(puma_occurrences)} Puma-like species with occurrences in 2015")

# sort by number of occurrences and print
sorted_occurrences = sorted(
    puma_occurrences.items(),
    key=lambda x: x[1]['numOccurrences'],
    reverse=True
)

for taxon_key, data in sorted_occurrences[:10]:
    print(f"{data['scientificName']} ({taxon_key}): {data['numOccurrences']} occurrences")

4. Create a simple map of Puma concolor (Linnaeus, 1771) (2435099) occurrences in 2015

In [ ]:
puma_tiles = maps.map(taxonKey=2435099, year=2015)
puma_tiles.plot()


5. Collect the detailed 2015 occurrence data for Puma concolor (Linnaeus, 1771) (2435099)

In [ ]:
puma_occurrence_details = occurrences.search(taxonKey=2435099, year=2015)

# Filter out only data with coordinates
puma_locations = [occ for occ in puma_occurrence_details['results']
                  if 'decimalLatitude' in occ and 'decimalLongitude' in occ]
print(f"Retrieved {len(puma_locations)} detailed occurrences with location data for Puma concolor in 2015")


for occ in puma_locations[:5]:
    print(f"- {occ['scientificName']} observed at ({occ['decimalLatitude']}, {occ['decimalLongitude']}) on {occ['eventDate']}")


6. Create plot with map overlay

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point
import geodatasets

# Convert to GeoDataFrame
points = [Point(occ['decimalLongitude'], occ['decimalLatitude']) for occ in puma_locations]
gdf = gpd.GeoDataFrame(puma_locations, geometry=points, crs="EPSG:4326")

# Load land boundaries
world = gpd.read_file(geodatasets.get_path('naturalearth.land'))

# Create plot
fig, ax = plt.subplots(figsize=(20, 12))

# Plot world boundaries
world.boundary.plot(ax=ax, linewidth=0.8, color='black', alpha=0.7)
world.plot(ax=ax, color='lightgray', alpha=0.3, edgecolor='black')

# Plot Puma locations
gdf.plot(ax=ax, color='darkred', markersize=8, alpha=0.6, label='Puma Locations')

ax.set_title('Puma concolor Observations in 2015')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend()
plt.grid(True, alpha=0.2, linestyle='--')
plt.tight_layout()
plt.show()